# 3. Follow the evidence

**Question:** Which observations demonstrate runtime behavior, and which establish security risk?

An invocation response, a content-filter block, an Application Insights trace, a Defender alert, and an Agent 365 risk signal are separate evidence. The repository has **not verified end-to-end Agent 365 ingestion or agent-attributed jailbreak risk**. This notebook does not invent a Defender alert from a blocked response or assume historical alert backfill.

Prompt agents have Foundry-managed collection. Hosted agents use the native hosting SDK and Microsoft OpenTelemetry. Neither path is automatically routed through the standalone APIM model API.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/governance').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from IPython.display import Markdown, display
from src.governance.evidence import synthetic_estate, load_snapshot, markdown_table

# Optional normalized CLI export; source events retain their original event times.
SNAPSHOT_PATH = None
estate = load_snapshot(Path(SNAPSHOT_PATH)) if SNAPSHOT_PATH else synthetic_estate()
rows = [row.model_dump(mode='json') for row in estate.evidence if row.category in ('control', 'security', 'visibility')]
display(Markdown(markdown_table(rows)))
assert not any(row['status'] == 'observed' and row['evidence_kind'] == 'unverified' for row in rows)

## Optional synthetic input tests against existing endpoints

Two gates separate paid inference from the jailbreak test. Both are false by default. Normal calls use fictional HR questions; the jailbreak option requests hidden instructions as a synthetic guardrail test. It can produce security events. It does **not** mark an identity compromised or alter any policy.

The wrapper invokes existing prompt/hosted endpoints with no provisioning and no local-search fallback. Request metadata may provide correlation IDs, but a request ID is not necessarily an OpenTelemetry trace ID. Model output and prompt text are not exported.

In [ ]:
ALLOW_PAID_INFERENCE = False
ALLOW_JAILBREAK_TEST = False

if ALLOW_PAID_INFERENCE:
    from dotenv import load_dotenv
    from src.governance.live import invoke_agent, invoke_gateway
    load_dotenv(ROOT / '.env')
    results = [
        invoke_gateway(allow_inference=True, jailbreak=ALLOW_JAILBREAK_TEST),
        invoke_agent('HRPolicyAgent', allow_inference=True, jailbreak=ALLOW_JAILBREAK_TEST),
        invoke_agent('hr-policy-agent', hosted=True, allow_inference=True, jailbreak=ALLOW_JAILBREAK_TEST),
    ]
    display(Markdown(markdown_table([row.model_dump(mode='json') for row in results])))
else:
    print('Inference and jailbreak tests disabled. No prompts sent.')

## Attribution and drill-through

| Evidence | Establishes | Does not establish |
| --- | --- | --- |
| Agent definition inventory | Agent exists in a project | Publication, ingestion, or active risk |
| HTTP filter response | Filter outcome for that request | Defender detection or Agent 365 risk |
| Correlated trace | Observed execution at the trace source | A security finding |
| Configuration preflight | Collection/permission prerequisites | Export delivery or historical backfill |
| Account-matched Defender source alert | Detection at the source account | Specific agent attribution or Agent 365 arrival |

The live collector exposes selected account-level Defender alert metadata without suspicious-prompt evidence. Agent 365 risk feeds, Purview findings, Sentinel incidents and agent-identity correlation remain explicit visibility gaps. Azure Government applicability must be checked separately from this Commercial demo.

Implementation reference: [Agent365Observability.md](../../docs/Agent365Observability.md).